# Frequency Stability Analysis## ObjectiveAssesses frequency behavior following generation lossProject: 39 Bus New England System - 2**Study Case: Study Cases 1. Power Flow****Objective:**- Assess frequency behavior following generation loss- Simulate disconnection of a generating unit- Perform simulations for Base Case and New Generation Case- Analyze frequency dynamics- Key Metrics: System frequency, Rate of Change of Frequency (RoCoF), Frequency nadir, Generator speed response- Outputs: Frequency and RoCoF plots, Generator speed vs. time, Comparative performance indicators---

## Step 1: Access PowerFactoryFirst, we need to set up the Python environment to access DIgSILENT PowerFactory.

In [ ]:
# ============================================================================# STEP 1: Access PowerFactory# ============================================================================import osos.environ["PATH"] = r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2" + os.environ["PATH"]import syssys.path.append(r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9")# Import powerfactoryimport powerfactory as pfapp = pf.GetApplication()  # Get the application# ============================================================================# STEP 2: Access and activate project# ============================================================================user = app.GetCurrentUser()project = app.ActivateProject("39 Bus New England System - 2")  # Activate the desired projectprj = app.GetActiveProject()print(f"Project activated: {prj.loc_name}")# ============================================================================# STEP 2.5: Activate study case (if needed)# ============================================================================# Try to activate the study case "Study Cases 1. Power Flow"try:    study_cases = prj.GetContents('*.IntCase')    for sc in study_cases:        if '1. Power Flow' in sc.loc_name or 'Power Flow' in sc.loc_name:            sc.Activate()            print(f"Study case activated: {sc.loc_name}")            breakexcept:    print("Note: Using default/active study case")# ============================================================================# STEP 3: Get all relevant objects (generators, buses)# ============================================================================# Create generator dictionarygenerators = app.GetCalcRelevantObjects('*.ElmSym')gen_dict = {}for gen in generators:    gen_dict[gen.loc_name] = gen# Create bus dictionarybuses = app.GetCalcRelevantObjects('*.ElmTerm')bus_dict = {}for bus in buses:    bus_dict[bus.loc_name] = busprint(f"Found {len(gen_dict)} generators and {len(bus_dict)} buses")# ============================================================================# STEP 4: Select generator for disconnection - Base Case# ============================================================================print("\n=== Setting up Generation Loss - Base Case ===")app.ResetCalculation()# Select a generator for disconnection (avoid the largest one for stability)disconnect_gen = Noneif len(gen_dict) > 0:    # Select first generator (or modify to select specific one)    disconnect_gen = list(gen_dict.values())[0]    print(f"Selected generator for disconnection: {disconnect_gen.loc_name}")        # Get initial power output    initial_power = disconnect_gen.GetAttribute('m:P:bus1')    print(f"Initial power output: {initial_power:.4f} MW")# Select a bus for frequency monitoring (e.g., Bus 16)monitor_bus = Noneif 'Bus 16' in bus_dict:    monitor_bus = bus_dict['Bus 16']elif len(bus_dict) > 0:    monitor_bus = list(bus_dict.values())[0]if disconnect_gen and monitor_bus:    # Create generator disconnection event    event_folder = app.GetFromStudyCase('IntEvt')    event_name = 'Generator Disconnection Base'        try:        # Create generator trip event        event_folder.CreateObject('EvtOut', event_name)        trip_event = event_folder.GetContents(event_name)[0] if hasattr(event_folder, 'GetContents') else None                if trip_event is None:            events = event_folder.GetContents()            trip_event = events[0] if events else None                if trip_event:            trip_event.time = 1.0  # Generator trips at 1 second            trip_event.p_target = disconnect_gen                        print(f"Generator disconnection event created")            print(f"Trip time: {trip_event.time} s")    except Exception as e:        print(f"Note: Trip event creation may need manual setup: {e}")        # Alternative: Set generator out of service        try:            disconnect_gen.outserv = 0  # Will be set to 1 at event time        except:            pass# ============================================================================# STEP 5: Set up results monitoring - Base Case# ============================================================================elmres = app.GetFromStudyCase('All calculations.ElmRes')elmres.Clear()# Add frequency monitoringif monitor_bus:    elmres.AddVariable(monitor_bus, 'm:fehz')  # Frequency in Hz# Add generator speed monitoringfor gen_name, gen in gen_dict.items():    try:        elmres.AddVariable(gen, 'm:speed')  # Generator speed    except:        passprint("Added monitoring for frequency and generator speeds")# ============================================================================# STEP 6: Run Dynamic Simulation - Base Case# ============================================================================print("\n=== Running Dynamic Simulation - Base Case ===")# Set initial conditionsini = app.GetFromStudyCase('ComInc')ini.Execute()print("Initial conditions calculated")# Run dynamic simulationsim = app.GetFromStudyCase('ComSim')sim.tstop = 30.0  # Simulation time: 30 seconds (longer for frequency analysis)sim.Execute()print(f"Dynamic simulation completed (t = 0 to {sim.tstop} s)")# ============================================================================# STEP 7: Export Base Case Results# ============================================================================import osscript_dir = os.path.dirname(os.path.abspath(__file__))comres = app.GetFromStudyCase('ComRes')comres.iopt_csel = 0comres.iopt_locn = 1comres.ciopt_head = 1comres.pResult = elmrescomres.ipt_exp = 6  # CSV filecomres.f_name = os.path.join(script_dir, 'frequency_stability_base_case.csv')comres.Execute()print(f"Base case results exported to: frequency_stability_base_case.csv")# ============================================================================# STEP 8: New Generation Case Frequency Stability# ============================================================================print("\n=== Setting up Generation Loss - New Generation Case ===")# NOTE: This section should be modified based on how new generation is addedapp.ResetCalculation()# Re-create trip event for new generation caseif disconnect_gen and monitor_bus:    event_folder = app.GetFromStudyCase('IntEvt')    event_name_new = 'Generator Disconnection New Gen'        try:        event_folder.CreateObject('EvtOut', event_name_new)        trip_event_new = event_folder.GetContents(event_name_new)[0] if hasattr(event_folder, 'GetContents') else None                if trip_event_new is None:            events = event_folder.GetContents()            trip_event_new = events[-1] if events else None                if trip_event_new:            trip_event_new.time = 1.0            trip_event_new.p_target = disconnect_gen                        print(f"Generator disconnection event created for New Generation Case")    except Exception as e:        print(f"Note: Trip event creation may need manual setup: {e}")# Set up results monitoringelmres.Clear()if monitor_bus:    elmres.AddVariable(monitor_bus, 'm:fehz')for gen_name, gen in gen_dict.items():    try:        elmres.AddVariable(gen, 'm:speed')    except:        pass# Set initial conditionsini.Execute()# Run dynamic simulationsim.tstop = 30.0sim.Execute()print(f"Dynamic simulation completed for New Generation Case (t = 0 to {sim.tstop} s)")# ============================================================================# STEP 9: Export New Generation Case Results# ============================================================================comres.f_name = os.path.join(script_dir, 'frequency_stability_new_gen_case.csv')comres.Execute()print(f"New generation case results exported to: frequency_stability_new_gen_case.csv")# ============================================================================# STEP 10: Calculate Key Metrics and Create Comparison Plots# ============================================================================try:    import matplotlib.pyplot as plt    import pandas as pd    import numpy as np        # Read CSV files    base_df = pd.read_csv(os.path.join(script_dir, 'frequency_stability_base_case.csv'))    new_gen_df = pd.read_csv(os.path.join(script_dir, 'frequency_stability_new_gen_case.csv'))        # Get time column    time_col = base_df.columns[0]        # Find frequency column    freq_cols_base = [col for col in base_df.columns if 'fehz' in col.lower() or 'frequency' in col.lower()]    freq_cols_new = [col for col in new_gen_df.columns if 'fehz' in col.lower() or 'frequency' in col.lower()]        if freq_cols_base and freq_cols_new:        # Calculate RoCoF (Rate of Change of Frequency)        base_df['RoCoF'] = np.gradient(base_df[freq_cols_base[0]], base_df[time_col])        new_gen_df['RoCoF'] = np.gradient(new_gen_df[freq_cols_new[0]], new_gen_df[time_col])                # Find frequency nadir (minimum frequency)        base_nadir = base_df[freq_cols_base[0]].min()        base_nadir_time = base_df.loc[base_df[freq_cols_base[0]].idxmin(), time_col]        new_gen_nadir = new_gen_df[freq_cols_new[0]].min()        new_gen_nadir_time = new_gen_df.loc[new_gen_df[freq_cols_new[0]].idxmin(), time_col]                print(f"\n=== Frequency Metrics ===")        print(f"Base Case - Nadir: {base_nadir:.4f} Hz at t = {base_nadir_time:.2f} s")        print(f"New Gen Case - Nadir: {new_gen_nadir:.4f} Hz at t = {new_gen_nadir_time:.2f} s")        print(f"Base Case - Max RoCoF: {base_df['RoCoF'].min():.4f} Hz/s")        print(f"New Gen Case - Max RoCoF: {new_gen_df['RoCoF'].min():.4f} Hz/s")                # Plot frequency response        fig, axes = plt.subplots(2, 1, figsize=(12, 10))                # Frequency plot        axes[0].plot(base_df[time_col], base_df[freq_cols_base[0]],                      label='Base Case', color='blue', linewidth=2)        axes[0].plot(new_gen_df[time_col], new_gen_df[freq_cols_new[0]],                      label='New Generation Case', color='red', linewidth=2)        axes[0].axhline(y=50.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Nominal (50 Hz)')        axes[0].set_xlabel('Time (s)', fontsize=12)        axes[0].set_ylabel('Frequency (Hz)', fontsize=12)        axes[0].set_title('System Frequency Response', fontsize=14, fontweight='bold')        axes[0].legend(fontsize=10)        axes[0].grid(True, alpha=0.3)                # RoCoF plot        axes[1].plot(base_df[time_col], base_df['RoCoF'],                     label='Base Case', color='blue', linewidth=2)        axes[1].plot(new_gen_df[time_col], new_gen_df['RoCoF'],                     label='New Generation Case', color='red', linewidth=2)        axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)        axes[1].set_xlabel('Time (s)', fontsize=12)        axes[1].set_ylabel('RoCoF (Hz/s)', fontsize=12)        axes[1].set_title('Rate of Change of Frequency (RoCoF)', fontsize=14, fontweight='bold')        axes[1].legend(fontsize=10)        axes[1].grid(True, alpha=0.3)                plt.tight_layout()        plot_path = os.path.join(script_dir, 'frequency_stability_plots.png')        plt.savefig(plot_path, dpi=300, bbox_inches='tight')        plt.close()        print(f"Static plots saved to: frequency_stability_plots.png")                # Export metrics to CSV        import csv        metrics_csv = os.path.join(script_dir, 'frequency_metrics_comparison.csv')        with open(metrics_csv, 'w', newline='') as f:            writer = csv.writer(f)            writer.writerow(['Metric', 'Base Case', 'New Generation Case', 'Difference'])            writer.writerow(['Frequency Nadir (Hz)', base_nadir, new_gen_nadir, new_gen_nadir - base_nadir])            writer.writerow(['Nadir Time (s)', base_nadir_time, new_gen_nadir_time, new_gen_nadir_time - base_nadir_time])            writer.writerow(['Max RoCoF (Hz/s)', base_df['RoCoF'].min(), new_gen_df['RoCoF'].min(),                            new_gen_df['RoCoF'].min() - base_df['RoCoF'].min()])        print(f"Frequency metrics exported to: frequency_metrics_comparison.csv")        # Plot generator speeds    speed_cols_base = [col for col in base_df.columns if 'speed' in col.lower()]    speed_cols_new = [col for col in new_gen_df.columns if 'speed' in col.lower()]        if speed_cols_base and speed_cols_new:        num_speed_plots = min(len(speed_cols_base), 3)        fig, axes = plt.subplots(num_speed_plots, 1, figsize=(14, 4 * num_speed_plots))        if num_speed_plots == 1:            axes = [axes]                for idx, (col_base, col_new) in enumerate(zip(speed_cols_base[:num_speed_plots], speed_cols_new[:num_speed_plots])):            ax = axes[idx] if idx < len(axes) else axes[-1]            ax.plot(base_df[time_col], base_df[col_base], label='Base Case', color='blue', linewidth=2)            ax.plot(new_gen_df[time_col], new_gen_df[col_new], label='New Generation Case', color='red', linewidth=2)            ax.set_xlabel('Time (s)', fontsize=10)            ax.set_ylabel('Speed (p.u.)', fontsize=10)            ax.set_title(f'Generator Speed Response: {col_base}', fontsize=11, fontweight='bold')            ax.legend()            ax.grid(True, alpha=0.3)                plt.tight_layout()        speed_plot_path = os.path.join(script_dir, 'generator_speed_comparison.png')        plt.savefig(speed_plot_path, dpi=300, bbox_inches='tight')        plt.close()        print("Generator speed comparison plot saved")        # Interactive Bokeh plots    try:        from bokeh.plotting import figure, output_file, save        from bokeh.models import ColumnDataSource, HoverTool        from bokeh.layouts import gridplot                output_file(os.path.join(script_dir, 'frequency_stability_interactive.html'))        plots = []                # Frequency and RoCoF plot        if freq_cols_base and freq_cols_new:            p1 = figure(width=900, height=400, title="System Frequency Response (Interactive)",                       x_axis_label="Time (s)", y_axis_label="Frequency (Hz)",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save")                        base_freq_source = ColumnDataSource(data=dict(                time=base_df[time_col],                freq=base_df[freq_cols_base[0]]            ))                        new_gen_freq_source = ColumnDataSource(data=dict(                time=new_gen_df[time_col],                freq=new_gen_df[freq_cols_new[0]]            ))                        p1.line('time', 'freq', source=base_freq_source, color='blue',                    line_width=2, legend_label='Base Case', alpha=0.8)            p1.line('time', 'freq', source=new_gen_freq_source, color='red',                    line_width=2, legend_label='New Generation Case', alpha=0.8)            p1.line([base_df[time_col].min(), base_df[time_col].max()], [50.0, 50.0],                    color='black', line_dash='dashed', line_width=1, legend_label='Nominal (50 Hz)')                        hover = p1.select_one(HoverTool)            hover.tooltips = [("Time", "@time{0.00} s"), ("Frequency", "@freq{0.000} Hz")]            p1.legend.location = "top_right"            plots.append(p1)                        # RoCoF plot            p2 = figure(width=900, height=400, title="Rate of Change of Frequency - RoCoF (Interactive)",                       x_axis_label="Time (s)", y_axis_label="RoCoF (Hz/s)",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save")                        base_rocof_source = ColumnDataSource(data=dict(                time=base_df[time_col],                rocof=base_df['RoCoF']            ))                        new_gen_rocof_source = ColumnDataSource(data=dict(                time=new_gen_df[time_col],                rocof=new_gen_df['RoCoF']            ))                        p2.line('time', 'rocof', source=base_rocof_source, color='blue',                    line_width=2, legend_label='Base Case', alpha=0.8)            p2.line('time', 'rocof', source=new_gen_rocof_source, color='red',                    line_width=2, legend_label='New Generation Case', alpha=0.8)            p2.line([base_df[time_col].min(), base_df[time_col].max()], [0, 0],                    color='black', line_dash='dashed', line_width=1)                        hover2 = p2.select_one(HoverTool)            hover2.tooltips = [("Time", "@time{0.00} s"), ("RoCoF", "@rocof{0.0000} Hz/s")]            p2.legend.location = "top_right"            plots.append(p2)                if plots:            grid = gridplot([plots], toolbar_location='right')            save(grid)            print(f"Interactive plots saved to: frequency_stability_interactive.html")    except Exception as e:        print(f"Note: Bokeh interactive plot creation failed: {e}")    except ImportError as e:    print(f"Note: Visualization libraries not available: {e}")    print("Install required packages: pip install matplotlib seaborn pandas bokeh numpy")except Exception as e:    print(f"Note: Error creating plots: {e}")# ============================================================================# STEP 11: Clean up# ============================================================================app.ResetCalculation()# Restore generatorif disconnect_gen:    try:        disconnect_gen.outserv = 0    except:        pass# Delete eventstry:    event_folder = app.GetFromStudyCase('IntEvt')    events = event_folder.GetContents()    for event in events:        if 'Generator Disconnection' in event.loc_name:            event.Delete()except:    passprint("\n=== Frequency Stability Analysis completed successfully ===")print(f"Results saved in: {script_dir}")